# 🧪 Testing Your Model
### D4BL Tutorial 4 of 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/William-Hill/d4bl_ai_agent/blob/main/notebooks/tutorials/04_testing_your_model.ipynb)

**What you'll learn:** How to evaluate your fine-tuned model against the base model, validate structured output, and apply D4BL's ship criteria.

**Time:** ~15 minutes | **Prerequisites:** GPU runtime (for inference) | **Dependencies:** unsloth, transformers

In [ ]:
!pip install -q unsloth transformers

import json

from unsloth import FastLanguageModel

## Loading Your Model

If you ran Notebook 3, your adapter is saved locally. If not, we provide a fallback to run with the base model for comparison purposes.

In [ ]:
import os

ADAPTER_PATH = "d4bl-query-parser-adapter"

if os.path.exists(ADAPTER_PATH):
    print(f"Found local adapter at {ADAPTER_PATH}")
else:
    print("No local adapter found.")
    print("Option 1: Run Notebook 3 first to train your own.")
    print("Option 2: Uncomment the line below to download a pre-trained adapter.")
    # !git clone https://huggingface.co/d4bl/query-parser-adapter {ADAPTER_PATH}
    ADAPTER_PATH = None

# Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Load adapter if available
if ADAPTER_PATH:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    print("Adapter loaded successfully!")
else:
    print("Running with base model only (no fine-tuning).")

FastLanguageModel.for_inference(model)

In [ ]:
TEST_PROMPTS = [
    "Why can't our kids breathe clean air in this neighborhood?",
    "What's the maternal mortality rate for Black women in Alabama?",
    "Show me poverty data for Mississippi",
    "Why do they keep building factories near our homes?",
    "Compare homeownership rates by race in Georgia",
]

SYSTEM_PROMPT = (
    "You are an AI assistant trained to support data justice and racial equity research "
    "following the Data for Black Lives (D4BL) methodology.\n\n"
    "Respond with ONLY valid JSON."
)

def generate_response(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=512, temperature=0.1, do_sample=True
    )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

print("Generating responses...\n")
results = []
for prompt in TEST_PROMPTS:
    response = generate_response(prompt)
    results.append({"prompt": prompt, "response": response})
    print(f"Q: {prompt}")
    print(f"A: {response[:200]}...")
    print()

## Validating Structured Output

For the query parser, every response should be valid JSON with specific fields. Let's check:

> **Note:** If you're running without an adapter (base model only), expect most validations to fail — the base model hasn't been trained to produce structured JSON with these fields. That's the whole point of fine-tuning!

In [ ]:
REQUIRED_FIELDS = {"entities", "search_queries", "data_sources", "community_framing"}

def validate_response(response_text):
    """Check if response is valid JSON with required fields."""
    try:
        parsed = json.loads(response_text)
    except json.JSONDecodeError:
        return {"valid_json": False, "has_fields": False, "parsed": None}

    has_fields = REQUIRED_FIELDS.issubset(set(parsed.keys()))
    return {"valid_json": True, "has_fields": has_fields, "parsed": parsed}

print("Validation Results:")
print("=" * 60)
valid_count = 0
for r in results:
    v = validate_response(r["response"])
    status = "✅" if v["valid_json"] and v["has_fields"] else "❌"
    if v["valid_json"] and v["has_fields"]:
        valid_count += 1
    print(f"{status} {r['prompt'][:50]}...")
    print(f"   Valid JSON: {v['valid_json']} | Required fields: {v['has_fields']}")

print(f"\nPass rate: {valid_count}/{len(results)} ({valid_count/len(results):.0%})")

## Ship Criteria

D4BL doesn't just ask "does it work?" — we have formal ship criteria:

| Metric | Threshold | What It Measures |
|--------|-----------|------------------|
| JSON validity rate | ≥ 90% | Model produces parseable structured output |
| Field completeness | ≥ 85% | All required fields present |
| Community framing detection | ≥ 70% | Recognizes community-voiced questions |
| P95 latency | < 1000ms | Fast enough for interactive use |

If any **blocking** criterion fails, the model doesn't ship. Non-blocking gaps are tracked but don't prevent deployment.

## ✏️ Exercise

1. Write 3 test prompts about a topic you care about — housing, environment, health, education.
2. Run them through `generate_response()` and validate the output.
3. For community-framed questions, check if `community_framing.detected` is `true`.

In [ ]:
# TODO: Add your own test prompts
# my_prompts = [
#     "...",
#     "...",
#     "...",
# ]
# for p in my_prompts:
#     response = generate_response(p)
#     print(f"Q: {p}")
#     print(f"A: {response}")
#     print(f"Valid: {validate_response(response)}")
#     print()

## Summary

You've tested a fine-tuned model against real prompts, validated structured output, and learned about D4BL's ship criteria. The model isn't just "good enough" — it meets explicit standards for JSON validity, field completeness, and community framing detection.

**Next:** [Notebook 5 — Making It Your Own](https://colab.research.google.com/github/William-Hill/d4bl_ai_agent/blob/main/notebooks/tutorials/05_making_it_your_own.ipynb) → Customize the model for your community's data.